In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


# Supplementary Figure 11

Clean analysis workflow for Supplementary Fig. 11a-c. The notebook starts from the original h5ad file, computes sample stress groups, calculates acid-base scores from log1p(CP10K)-normalized expression, compares high- and low-stress samples, and exports one PDF figure plus one source-data/statistics workbook.

In [ ]:

from pathlib import Path
import warnings

import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

H5AD_PATH = Path("/data/beifen/zhongmin/泛癌S/h5ad/189数据/combined_adata_inner.h5ad")
OUTDIR = Path("/work/zhongmin/daima/daima/slide-tag/整理代码/补充图11")
OUTDIR.mkdir(parents=True, exist_ok=True)

FIGURE_A_PDF = OUTDIR / "figureS11a.pdf"
FIGURE_B_PDF = OUTDIR / "figureS11b.pdf"
FIGURE_C_PDF = OUTDIR / "figureS11c.pdf"
SOURCE_DATA_FILE = OUTDIR / "Source_Data_Supplementary_Fig11.xlsx"

SAMPLE_COL = "sample"
PATHOLOGY_COL = "group"
CELLTYPE1_COL = "celltype_1"
CELLTYPE2_COL = "celltype_2"
CELLTYPE3_COL = "celltype_3"
TISSUE_ORGAN_COL = "tissue_organ"
STRESS_GROUP_COL = "stress_group"

HIGH_LABEL = "High_stress"
LOW_LABEL = "Low_stress"
MID_LABEL = "Mid_stress"
NO_IMMUNE_LABEL = "No_immune"
PLOT_GROUPS = [HIGH_LABEL, LOW_LABEL]
PLOT_GROUP_LABELS = {HIGH_LABEL: "High Stress", LOW_LABEL: "Low Stress"}
HIGH_STRESS_THRESHOLD = 0.20

IMMUNE_CT1 = {
    "T cells", "Monocyte_macrophage", "Plasma cells", "B cells", "NK cells",
    "DC", "Mast cells", "pDC", "Neutrophils",
}
STRESS_KEYWORDS = ["HSPA1A", "HSP90AA1", "Stress"]

ACIDIFYING_GENES = [
    "SLC2A1", "HK2", "PFKP", "ALDOA", "GAPDH", "ENO1", "PKM", "LDHA", "PDK1",
    "SLC16A3", "SLC16A1", "SLC9A1", "CA9", "CA12", "ATP6V1A", "ATP6V0A1", "ATP6V1B2",
]
BUFFERING_GENES = ["SLC4A4", "SLC4A2"]

TARGET_IMMUNE_STATES = [
    "Exhausted CD8 TEM",
    "Exhausted CD8_Cycling",
    "Exhausted_Stress response CD8 TEM",
    "Precursor exhausted CD8 TCM",
    "Precursor exhausted CD8 TEM",
    "Treg",
]

TARGET_SUM = 1e4

PATHOLOGY_COLORS = {
    "Tumor_metastasis": "#ff8a95",
    "Tumor": "#de2d26",
    "Precancerous condition": "#74c476",
    "Normal adjacent tissue": "#4daf4a",
    "Normal": "#fdae6b",
    "Inflammation": "#fd8d3c",
    "Blood_Tumor": "#9ecae1",
    "Autoimmune diseases": "#3182bd",
}
PALETTE = {HIGH_LABEL: "#E66B2E", LOW_LABEL: "#2C7FB8"}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 7,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "legend.frameon": False,
})


## Normalization note

For panel b, the full h5ad object is loaded into memory and normalized once with `sc.pp.normalize_total(adata, target_sum=1e4)`, followed once by `sc.pp.log1p(adata)`. Acidifying, buffering and net scores are then computed from this normalized expression matrix.

In [ ]:

def standardize_annotations(obs: pd.DataFrame) -> pd.DataFrame:
    obs = obs.copy()
    for col in [CELLTYPE1_COL, CELLTYPE2_COL, CELLTYPE3_COL]:
        obs[col] = obs[col].astype(str)

    mask_low_cnv = obs[CELLTYPE3_COL].astype("string").str.contains("Malignant cells_low_cnv", na=False)
    obs.loc[mask_low_cnv, CELLTYPE2_COL] = "Epithelial cells_low_cnv"
    obs.loc[mask_low_cnv, CELLTYPE1_COL] = "Epithelial cells"

    ct3_to_ct2 = {
        "RNR2_DC2": "DC2",
        "stomach_epithelial cell": "epithelial cell",
        "Esophagus_epithelial cell": "epithelial cell",
        "foveolar cell of stomach": "epithelial cell",
        "intestine_epithelial cell": "epithelial cell",
        "mucous neck cell": "epithelial cell",
        "Plasmablast": "Plasmablast",
        "Pericyte": "Pericyte",
        "SMC": "SMC",
    }
    for ct3, ct2 in ct3_to_ct2.items():
        obs.loc[obs[CELLTYPE3_COL].eq(ct3), CELLTYPE2_COL] = ct2

    ct2_to_ct1 = {
        "Plasmablast": "B cells",
        "GCB": "B cells",
        "DC1": "DC",
        "DC_Cycling": "DC",
        "LAMP3_DC": "DC",
        "Langerhans": "DC",
        "DC2": "DC",
        "pDC": "pDC",
        "Mast cells": "Mast cells",
        "CD16 monocyte": "Monocyte_macrophage",
        "CD14 monocyte": "Monocyte_macrophage",
        "Macrophage": "Monocyte_macrophage",
        "Alveolar macrophage": "Monocyte_macrophage",
        "CD14CD16 monocyte": "Monocyte_macrophage",
        "Neutrophils": "Neutrophils",
        "epithelial cell": "Epithelial cells",
        "gland cells": "Epithelial cells",
        "Goblet": "Epithelial cells",
    }
    for ct2, ct1 in ct2_to_ct1.items():
        obs.loc[obs[CELLTYPE2_COL].eq(ct2), CELLTYPE1_COL] = ct1

    ct1_recode = {
        "NKT cells": "T cells",
        "SMC": "Fibroblast cells",
        "Pericyte": "Fibroblast cells",
        "epithelial cell": "Epithelial cells",
        "Other epithelial": "Epithelial cells",
        "Other": "Epithelial cells",
        "Specialized cells": "Epithelial cells",
        "Multiciliated cells": "Epithelial cells",
        "Basal cells": "Epithelial cells",
        "Neuroendocrine cells": "Epithelial cells",
    }
    obs[CELLTYPE1_COL] = obs[CELLTYPE1_COL].replace(ct1_recode)
    return obs


def classify_stress_groups(obs: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    obs = obs.copy()
    immune_mask = obs[CELLTYPE1_COL].astype("string").isin(IMMUNE_CT1)

    ct3_up = obs[CELLTYPE3_COL].astype("string").str.upper()
    pattern = "(" + "|".join(k.upper().replace(" ", r"\s*") for k in STRESS_KEYWORDS) + ")"
    stress_mask = ct3_up.str.contains(pattern, regex=True, na=False)
    stress_in_immune_mask = immune_mask & stress_mask

    sample = obs[SAMPLE_COL].astype("string")
    immune_n = immune_mask.groupby(sample).sum().astype(int)
    stress_n = stress_in_immune_mask.groupby(sample).sum().astype(int)
    stress_ratio = stress_n / immune_n.replace(0, np.nan)

    sample_summary = pd.DataFrame({
        "immune_cell_count": immune_n,
        "stress_immune_cell_count": stress_n,
        "stress_immune_fraction": stress_ratio,
    })

    def classify(row):
        if row["immune_cell_count"] == 0:
            return NO_IMMUNE_LABEL
        if row["stress_immune_cell_count"] == 0:
            return LOW_LABEL
        if row["stress_immune_fraction"] > HIGH_STRESS_THRESHOLD:
            return HIGH_LABEL
        return MID_LABEL

    sample_summary[STRESS_GROUP_COL] = sample_summary.apply(classify, axis=1)
    obs[STRESS_GROUP_COL] = sample.map(sample_summary[STRESS_GROUP_COL]).astype("category")
    sample_summary = sample_summary.reset_index().rename(columns={SAMPLE_COL: "sample"})
    return obs, sample_summary


def compare_groups(df: pd.DataFrame, value_cols: list[str], feature_col: str, correction_scope: str) -> pd.DataFrame:
    rows = []
    for feature in value_cols:
        v_high = df.loc[df[STRESS_GROUP_COL].eq(HIGH_LABEL), feature].dropna().to_numpy(dtype=float)
        v_low = df.loc[df[STRESS_GROUP_COL].eq(LOW_LABEL), feature].dropna().to_numpy(dtype=float)
        if len(v_high) >= 3 and len(v_low) >= 3:
            stat, pval = mannwhitneyu(v_high, v_low, alternative="two-sided")
        else:
            stat, pval = np.nan, np.nan
        rows.append({
            feature_col: feature,
            "n_high": len(v_high),
            "n_low": len(v_low),
            "mean_high": np.nanmean(v_high) if len(v_high) else np.nan,
            "mean_low": np.nanmean(v_low) if len(v_low) else np.nan,
            "median_high": np.nanmedian(v_high) if len(v_high) else np.nan,
            "median_low": np.nanmedian(v_low) if len(v_low) else np.nan,
            "difference_mean_high_minus_low": (np.nanmean(v_high) - np.nanmean(v_low)) if len(v_high) and len(v_low) else np.nan,
            "difference_median_high_minus_low": (np.nanmedian(v_high) - np.nanmedian(v_low)) if len(v_high) and len(v_low) else np.nan,
            "mannwhitney_u": stat,
            "p_value": pval,
            "multiple_testing_correction_scope": correction_scope,
        })
    stats = pd.DataFrame(rows)
    stats["fdr_bh"] = np.nan
    mask = stats["p_value"].notna()
    if mask.any():
        stats.loc[mask, "fdr_bh"] = multipletests(stats.loc[mask, "p_value"], method="fdr_bh")[1]
    return stats


def make_source_statistics(stats_df: pd.DataFrame, panel: str, feature_col: str, values_sheet: str, value_description: str) -> pd.DataFrame:
    out = pd.DataFrame({
        "figure": "Supplementary Fig. 11",
        "panel": panel,
        "feature": stats_df[feature_col],
        "group1": "High Stress",
        "group2": "Low Stress",
        "n_group1": stats_df["n_high"],
        "n_group2": stats_df["n_low"],
        "n_unit": "sample",
        "group1_mean": stats_df["mean_high"],
        "group2_mean": stats_df["mean_low"],
        "group1_median": stats_df["median_high"],
        "group2_median": stats_df["median_low"],
        "effect_size": stats_df["difference_mean_high_minus_low"],
        "effect_size_definition": "group1_mean_minus_group2_mean",
        "statistical_test": "two-sided Mann-Whitney U test",
        "raw_p_value": stats_df["p_value"],
        "adjusted_p_value_fdr": stats_df["fdr_bh"],
        "multiple_testing_correction": "Benjamini-Hochberg",
        "multiple_testing_correction_scope": stats_df["multiple_testing_correction_scope"],
        "values_sheet": values_sheet,
        "value_description": value_description,
    })
    return out


def p_to_stars(p):
    if pd.isna(p):
        return "n.s."
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "n.s."


def add_sig_bar(ax, y, h, text, x1=0, x2=1):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=0.9, color="#263238")
    ax.text((x1 + x2) / 2, y + h, text, ha="center", va="bottom", fontsize=7, color="#263238")


In [ ]:
adata = ad.read_h5ad(H5AD_PATH)
print(adata)

# Normalize the complete expression matrix once before score calculation.
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
sc.pp.log1p(adata)

required_obs = [SAMPLE_COL, PATHOLOGY_COL, CELLTYPE1_COL, CELLTYPE2_COL, CELLTYPE3_COL, TISSUE_ORGAN_COL]
missing = [c for c in required_obs if c not in adata.obs.columns]
if missing:
    raise KeyError(f"Missing obs columns: {missing}")

obs = adata.obs[required_obs].copy()
obs = standardize_annotations(obs)
analysis_mask = ~obs[TISSUE_ORGAN_COL].astype(str).eq("Blood")
obs_analysis, sample_stress_summary = classify_stress_groups(obs.loc[analysis_mask].copy())

obs[STRESS_GROUP_COL] = pd.NA
obs.loc[analysis_mask, STRESS_GROUP_COL] = obs_analysis[STRESS_GROUP_COL].astype(str).values

sample_stress_summary[STRESS_GROUP_COL].value_counts(dropna=False)


In [ ]:
def compute_acid_base_scores(adata, obs: pd.DataFrame) -> pd.DataFrame:
    """Compute acid-base scores from the normalized full AnnData matrix."""
    genes = list(dict.fromkeys(ACIDIFYING_GENES + BUFFERING_GENES))
    present = [g for g in genes if g in adata.var_names]
    missing = [g for g in genes if g not in adata.var_names]
    if missing:
        print(f"Missing genes ignored: {missing}")

    acid_present = [g for g in ACIDIFYING_GENES if g in present]
    buffer_present = [g for g in BUFFERING_GENES if g in present]
    if not acid_present or not buffer_present:
        raise ValueError("At least one acidifying and one buffering gene must be present.")

    gene_idx = np.array([adata.var_names.get_loc(g) for g in present], dtype=int)
    acid_pos = np.array([present.index(g) for g in acid_present], dtype=int)
    buffer_pos = np.array([present.index(g) for g in buffer_present], dtype=int)

    group_values = obs[STRESS_GROUP_COL].astype(str).to_numpy()
    sample_values = obs[SAMPLE_COL].astype(str).to_numpy()
    mask_hl = np.isin(group_values, PLOT_GROUPS)

    X_gene = adata.X[:, gene_idx]
    if sp.issparse(X_gene):
        X_gene = X_gene.toarray()
    else:
        X_gene = np.asarray(X_gene)

    X_gene = X_gene[mask_hl, :]
    acid = np.nanmean(X_gene[:, acid_pos], axis=1)
    buffer = np.nanmean(X_gene[:, buffer_pos], axis=1)
    net = acid - buffer

    cell_scores = pd.DataFrame({
        SAMPLE_COL: sample_values[mask_hl],
        STRESS_GROUP_COL: group_values[mask_hl],
        "Acidifying_score": acid,
        "Buffering_score": buffer,
        "Net_score": net,
    })

    sample_scores = (
        cell_scores
        .groupby([SAMPLE_COL, STRESS_GROUP_COL], observed=True)[["Acidifying_score", "Buffering_score", "Net_score"]]
        .mean()
        .reset_index()
    )
    return sample_scores


def compute_immune_state_percentages(obs: pd.DataFrame) -> pd.DataFrame:
    immune_obs = obs.loc[obs[CELLTYPE1_COL].isin(IMMUNE_CT1), [SAMPLE_COL, STRESS_GROUP_COL, CELLTYPE3_COL]].copy()
    total_counts = immune_obs.groupby(SAMPLE_COL).size()
    target_obs = immune_obs[immune_obs[CELLTYPE3_COL].isin(TARGET_IMMUNE_STATES)]
    target_counts = pd.crosstab(target_obs[SAMPLE_COL], target_obs[CELLTYPE3_COL])
    target_counts = target_counts.reindex(total_counts.index, fill_value=0)
    for state in TARGET_IMMUNE_STATES:
        if state not in target_counts.columns:
            target_counts[state] = 0
    target_counts = target_counts[TARGET_IMMUNE_STATES]
    percentages = target_counts.div(total_counts, axis=0).mul(100).fillna(0)
    sample_group = obs[[SAMPLE_COL, STRESS_GROUP_COL]].drop_duplicates(SAMPLE_COL).set_index(SAMPLE_COL)[STRESS_GROUP_COL]
    percentages = percentages.merge(sample_group, left_index=True, right_index=True)
    percentages = percentages[percentages[STRESS_GROUP_COL].isin(PLOT_GROUPS)].copy()
    percentages.index.name = SAMPLE_COL
    return percentages.reset_index()


In [ ]:

# Panel a source data
sample_tbl = (
    obs[[SAMPLE_COL, STRESS_GROUP_COL, PATHOLOGY_COL]]
    .dropna()
    .drop_duplicates(subset=[SAMPLE_COL])
)
sample_tbl = sample_tbl[sample_tbl[STRESS_GROUP_COL].isin(PLOT_GROUPS)].copy()
pathology_counts = pd.crosstab(sample_tbl[STRESS_GROUP_COL], sample_tbl[PATHOLOGY_COL]).reindex(PLOT_GROUPS).fillna(0)
pathology_order = [c for c in PATHOLOGY_COLORS if c in pathology_counts.columns] + [c for c in pathology_counts.columns if c not in PATHOLOGY_COLORS]
pathology_counts = pathology_counts[pathology_order]
pathology_proportions = pathology_counts.div(pathology_counts.sum(axis=1), axis=0).fillna(0)

panel_a_plotted_values = (
    pathology_counts
    .stack()
    .rename("sample_count")
    .reset_index()
    .rename(columns={STRESS_GROUP_COL: "stress_group", PATHOLOGY_COL: "pathological_state"})
)
panel_a_props = (
    pathology_proportions
    .stack()
    .rename("proportion")
    .reset_index()
    .rename(columns={STRESS_GROUP_COL: "stress_group", PATHOLOGY_COL: "pathological_state"})
)
panel_a_plotted_values = panel_a_plotted_values.merge(panel_a_props, on=["stress_group", "pathological_state"], how="left")
panel_a_plotted_values.insert(0, "panel", "a")
panel_a_plotted_values.insert(0, "figure", "Supplementary Fig. 11")
panel_a_plotted_values["n_unit"] = "sample"

# Panel b source data and statistics
acid_base_scores = compute_acid_base_scores(adata, obs)
acid_base_metrics = ["Acidifying_score", "Buffering_score", "Net_score"]
acid_base_stats = compare_groups(
    acid_base_scores,
    acid_base_metrics,
    "metric",
    correction_scope="Panel b: three acid-base metrics",
)
panel_b_plotted_values = acid_base_scores.melt(
    id_vars=[SAMPLE_COL, STRESS_GROUP_COL],
    value_vars=acid_base_metrics,
    var_name="feature",
    value_name="plotted_value",
)
panel_b_plotted_values.insert(0, "panel", "b")
panel_b_plotted_values.insert(0, "figure", "Supplementary Fig. 11")
panel_b_plotted_values["n_unit"] = "sample"
panel_b_plotted_values["value_description"] = "Sample-level mean log1p(CP10K)-normalized expression score"

# Panel c source data and statistics
immune_state_percentages = compute_immune_state_percentages(obs)
immune_state_stats = compare_groups(
    immune_state_percentages,
    TARGET_IMMUNE_STATES,
    "cell_state",
    correction_scope="Panel c: six immune-cell states",
)
panel_c_plotted_values = immune_state_percentages.melt(
    id_vars=[SAMPLE_COL, STRESS_GROUP_COL],
    value_vars=TARGET_IMMUNE_STATES,
    var_name="feature",
    value_name="plotted_value",
)
panel_c_plotted_values.insert(0, "panel", "c")
panel_c_plotted_values.insert(0, "figure", "Supplementary Fig. 11")
panel_c_plotted_values["n_unit"] = "sample"
panel_c_plotted_values["value_description"] = "Percentage of immune cells assigned to the indicated cell state in each sample"

source_statistics = pd.concat([
    make_source_statistics(
        acid_base_stats,
        panel="b",
        feature_col="metric",
        values_sheet="S11b_plotted_values",
        value_description="Sample-level mean log1p(CP10K)-normalized expression score",
    ),
    make_source_statistics(
        immune_state_stats,
        panel="c",
        feature_col="cell_state",
        values_sheet="S11c_plotted_values",
        value_description="Sample-level percentage of immune cells in the indicated cell state",
    ),
], ignore_index=True)

gene_set_table = pd.DataFrame({
    "figure": ["Supplementary Fig. 11"] * (len(ACIDIFYING_GENES) + len(BUFFERING_GENES)),
    "panel": ["b"] * (len(ACIDIFYING_GENES) + len(BUFFERING_GENES)),
    "score": ["Acidifying_score"] * len(ACIDIFYING_GENES) + ["Buffering_score"] * len(BUFFERING_GENES),
    "gene": ACIDIFYING_GENES + BUFFERING_GENES,
})

readme_table = pd.DataFrame({
    "field": [
        "figure",
        "normalization",
        "group_definition",
        "panel_a_values",
        "panel_b_values",
        "panel_c_values",
        "statistical_test",
        "p_value_adjustment",
    ],
    "description": [
        "Supplementary Fig. 11",
        "Scores in panel b were computed from expression normalized with sc.pp.normalize_total(target_sum=1e4) followed by sc.pp.log1p.",
        "High-stress samples contain more than 20% stressed immune cells; low-stress samples contain no stressed immune cells.",
        "Pathological-state composition among high- and low-stress samples.",
        "Sample-level acidifying, buffering and net scores.",
        "Sample-level percentages of selected immune-cell states.",
        "Two-sided Mann-Whitney U test comparing high- and low-stress samples.",
        "Benjamini-Hochberg correction within each panel-specific metric set.",
    ],
})

print("Stress groups:")
print(sample_stress_summary[STRESS_GROUP_COL].value_counts(dropna=False))
print("\nAcid-base statistics:")
display(acid_base_stats)
print("\nImmune-state statistics:")
display(immune_state_stats)


In [ ]:
def plot_panel_a(ax):
    x = np.arange(len(PLOT_GROUPS))
    bottom = np.zeros(len(PLOT_GROUPS))
    for state in pathology_order:
        vals = pathology_proportions[state].to_numpy(dtype=float)
        ax.bar(
            x, vals, bottom=bottom, width=0.62,
            color=PATHOLOGY_COLORS.get(state, "#bdbdbd"),
            edgecolor="white", linewidth=0.45, label=state,
        )
        for xi, bi, vi in zip(x, bottom, vals):
            if vi >= 0.035:
                ax.text(xi, bi + vi / 2, f"{vi * 100:.1f}%", ha="center", va="center", fontsize=5.5)
        bottom += vals
    ax.set_xticks(x)
    ax.set_xticklabels([PLOT_GROUP_LABELS[g] for g in PLOT_GROUPS], fontsize=6.5)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Proportion of samples", fontsize=6.5)
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
    ax.set_title("Pathological-state composition by stress group", fontsize=7.2, pad=6)
    ax.grid(axis="y", linestyle="--", linewidth=0.45, alpha=0.25)
    ax.set_axisbelow(True)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], title="Pathological state", loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=5.4, title_fontsize=5.8)


def plot_box(ax, df, y_col, stats_df, feature_col, y_label, title_label):
    sns.boxplot(
        data=df, x=STRESS_GROUP_COL, y=y_col, order=PLOT_GROUPS, ax=ax,
        palette=PALETTE, width=0.52, linewidth=0.8, showfliers=False,
        boxprops={"alpha": 0.35, "edgecolor": "#263238"},
        whiskerprops={"color": "#263238", "linewidth": 0.8},
        capprops={"color": "#263238", "linewidth": 0.8},
        medianprops={"color": "#263238", "linewidth": 1.0},
    )
    sns.stripplot(
        data=df, x=STRESS_GROUP_COL, y=y_col, order=PLOT_GROUPS, ax=ax,
        palette=PALETTE, size=1.8, jitter=0.22, alpha=0.9, edgecolor="white", linewidth=0.2,
    )
    stat_row = stats_df.loc[stats_df[feature_col].eq(y_col)].iloc[0]
    qval = stat_row["fdr_bh"]
    stars = p_to_stars(qval)
    y_min = df[y_col].min()
    y_max = df[y_col].max()
    y_range = y_max - y_min if y_max != y_min else 1.0
    if stars != "n.s.":
        add_sig_bar(ax, y_max + y_range * 0.12, y_range * 0.05, stars)
    ax.set_title(f"{title_label}\n(FDR={qval:.3g})", fontsize=6.5, fontweight="bold", pad=7)
    ax.set_xlabel("")
    ax.set_ylabel(y_label, fontsize=6.2)
    ax.set_xticklabels([PLOT_GROUP_LABELS[g] for g in PLOT_GROUPS], fontsize=6.0)
    ax.grid(axis="y", linestyle="--", linewidth=0.45, alpha=0.28)
    ax.set_axisbelow(True)
    ax.set_ylim(top=y_max + y_range * 0.35)


In [ ]:

# Save panel a.
fig_a, ax_a = plt.subplots(figsize=(4.9, 3.0), dpi=300)
plot_panel_a(ax_a)
ax_a.text(-0.22, 1.08, "a", transform=ax_a.transAxes, fontsize=8, fontweight="bold", va="top")
fig_a.subplots_adjust(right=0.68)
fig_a.savefig(FIGURE_A_PDF, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURE_A_PDF}")

# Save panel b.
metric_labels = {
    "Acidifying_score": "Acidifying Score\n(Mean log1p CP10K)",
    "Buffering_score": "Buffering Score\n(Mean log1p CP10K)",
    "Net_score": "Net Score\n(Acid - Buffer)",
}
fig_b, axes_b = plt.subplots(1, 3, figsize=(6.8, 2.6), dpi=300)
for i, metric in enumerate(acid_base_metrics):
    plot_box(axes_b[i], acid_base_scores, metric, acid_base_stats, "metric", metric_labels[metric], metric.replace("_", " "))
    if i == 0:
        axes_b[i].text(-0.35, 1.08, "b", transform=axes_b[i].transAxes, fontsize=8, fontweight="bold", va="top")
fig_b.subplots_adjust(wspace=0.65)
fig_b.savefig(FIGURE_B_PDF, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURE_B_PDF}")

# Save panel c.
fig_c, axes_c = plt.subplots(1, len(TARGET_IMMUNE_STATES), figsize=(12.2, 2.75), dpi=300)
for i, state in enumerate(TARGET_IMMUNE_STATES):
    plot_box(axes_c[i], immune_state_percentages, state, immune_state_stats, "cell_state", "% of immune cells", state)
    if i == 0:
        axes_c[i].text(-0.35, 1.08, "c", transform=axes_c[i].transAxes, fontsize=8, fontweight="bold", va="top")
fig_c.subplots_adjust(wspace=0.75)
fig_c.savefig(FIGURE_C_PDF, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURE_C_PDF}")


In [ ]:

with pd.ExcelWriter(SOURCE_DATA_FILE, engine="openpyxl") as writer:
    readme_table.to_excel(writer, sheet_name="README", index=False)
    sample_stress_summary.to_excel(writer, sheet_name="sample_stress_groups", index=False)
    panel_a_plotted_values.to_excel(writer, sheet_name="S11a_plotted_values", index=False)
    panel_b_plotted_values.to_excel(writer, sheet_name="S11b_plotted_values", index=False)
    panel_c_plotted_values.to_excel(writer, sheet_name="S11c_plotted_values", index=False)
    source_statistics.to_excel(writer, sheet_name="statistics", index=False)
    gene_set_table.to_excel(writer, sheet_name="S11b_gene_sets", index=False)

print(f"Saved: {SOURCE_DATA_FILE}")
